In [ ]:
# Setup
import os, json, re
from pathlib import Path
from datetime import datetime

# Detect environment
IS_COLAB = False
IS_KAGGLE = False
ENV_NAME = "Local"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    ENV_NAME = "Colab"
    BASE_PATH = Path('/content/drive/MyDrive/AGoT-ReAct/Math Performance')
except Exception:
    # Check if Kaggle
    if os.path.exists('/kaggle/working'):
        IS_KAGGLE = True
        ENV_NAME = "Kaggle"
        BASE_PATH = Path('/kaggle/working')
    else:
        BASE_PATH = Path(r'f:\Data Science\DS\7th Semester\ML\Project\AGoT-ReAct\Math Performance')

print(f"{ENV_NAME} Environment | {BASE_PATH}")

# Check GPU availability
try:
    import torch
    if torch.cuda.is_available():
        print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("⚠️ No GPU detected - model will run on CPU (slower)")
except Exception:
    print("⚠️ PyTorch not installed - installing dependencies...")

# Install minimal deps
if IS_COLAB or IS_KAGGLE:
    os.system('pip install -q transformers torch accelerate datasets tqdm beautifulsoup4 requests bitsandbytes')
else:
    print("Installing dependencies locally...")
    os.system('pip install -q transformers torch accelerate datasets tqdm beautifulsoup4 requests')

Local | f:\Data Science\DS\7th Semester\ML\Project\AGoT-ReAct\Math Performance


In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# Paths & config
OUTPUT_DIR = BASE_PATH / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GPQA_OUTPUT_PATH = OUTPUT_DIR / 'gpqa_agot_react_results.jsonl'
GPQA_TRACES_PATH = OUTPUT_DIR / 'gpqa_agot_react_detailed_traces.jsonl'
GPQA_METRICS_PATH = OUTPUT_DIR / 'gpqa_agot_metrics.json'
GPQA_CUMULATIVE_PATH = OUTPUT_DIR / 'gpqa_agot_cumulative_metrics.json'
GPQA_CHECKPOINT_PATH = OUTPUT_DIR / 'gpqa_agot_checkpoint.json'

# Qwen2 defaults (override via env MODEL_NAME / CPU_FALLBACK_MODEL)
DEFAULT_MODEL = 'Qwen/Qwen2-7B-Instruct'
CPU_FALLBACK_MODEL = os.getenv('CPU_FALLBACK_MODEL', 'Qwen/Qwen2-1.5B-Instruct')
MODEL_NAME = os.getenv('MODEL_NAME', DEFAULT_MODEL)
AGOT_MAX_DEPTH = 3
AGOT_TOP_K = 3
AGOT_NUM_ATTEMPTS = 3
REACT_MAX_STEPS = 2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Available GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Require GPU to run.
if device != 'cuda':
    raise RuntimeError("CUDA GPU not detected. Enable a GPU runtime (e.g., T4) and restart the notebook.")

# Enable faster math paths on GPU
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"Loading {MODEL_NAME} from HuggingFace...")
print("This may take a few minutes on first run...")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

    print("Loading model in FP16 distributed across available GPUs...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map='auto',  # Automatically distribute across all available GPUs
        low_cpu_mem_usage=True,
        attn_implementation='sdpa',  # Use PyTorch's Scaled Dot Product Attention (no flash_attn needed)
    )
    model.eval()
    print("✓ Model loaded successfully on GPU")

except Exception as e:
    print(f"⚠️ Error loading model: {e}")
    print("Make sure you have enough disk space and VRAM; ensure GPU is enabled.")
    raise

print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Self-consistency: {AGOT_NUM_ATTEMPTS} attempts with majority voting")

Model: gpt-4o-mini
Output dir: f:\Data Science\DS\7th Semester\ML\Project\AGoT-ReAct\Math Performance\outputs
Ready!


In [ ]:
# ============= SHARED IMPORTS =============
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import asyncio

# Note: AGOTNode, AGOTGraph, and llm_generate are defined in Cell 11 (AGoT Engine)
# with domain analysis and verification capabilities. 
# They will be available after Cell 11 executes.

print("✓ Setup: Imports ready. Core functions will be loaded in Cell 11 (AGoT Engine).")

In [ ]:
import json
from pathlib import Path

# ============= GOOGLE DRIVE INTEGRATION =============
# Auto-detect and mount Google Drive (Kaggle + Colab compatible)

DRIVE_CONNECTED = False
DRIVE_OUTPUT_DIR = None
DRIVE_MOUNT_POINT = None

# Try Colab first
try:
    from google.colab import drive
    print("🔌 Attempting Google Drive connection (Colab)...")
    drive.mount('/content/drive', force_remount=True)
    DRIVE_MOUNT_POINT = Path('/content/drive/MyDrive/AGoT-ReAct-Outputs')
    DRIVE_MOUNT_POINT.mkdir(parents=True, exist_ok=True)
    DRIVE_OUTPUT_DIR = DRIVE_MOUNT_POINT
    DRIVE_CONNECTED = True
    print(f"✅ Google Drive connected (Colab)! Output directory: {DRIVE_OUTPUT_DIR}")
except Exception as e:
    print(f"⚠️ Colab not detected, trying Kaggle+PyDrive2...")
    
    # Try Kaggle with PyDrive2
    try:
        from pydrive2.auth import GoogleAuth
        from pydrive2.drive import GoogleDrive
        
        # For Kaggle, use user-uploaded credentials or interactive auth
        gauth = GoogleAuth()
        
        # Try to load saved credentials first
        if os.path.exists('mycreds.txt'):
            gauth.LoadCredentialsFile("mycreds.txt")
        
        if gauth.credentials is None:
            print("\n📌 KAGGLE DRIVE SETUP:")
            print("1. Go to https://myaccount.google.com/apppasswords")
            print("2. Generate an app password for 'Drive'")
            print("3. Run: gauth.LocalWebserverAuth()")
            print("   (This will open a browser for authentication)")
            
            # Attempt local webserver authentication
            try:
                gauth.LocalWebserverAuth()
                gauth.SaveCredentialsFile("mycreds.txt")
            except Exception as auth_err:
                print(f"⚠️ Could not authenticate: {str(auth_err)[:100]}")
                print("   Falling back to local-only mode (no Drive backup)")
                gauth = None
        
        if gauth and gauth.credentials is not None:
            drive = GoogleDrive(gauth)
            
            # Create or get AGoT-ReAct-Outputs folder
            folder_list = drive.ListFile({'q': "title='AGoT-ReAct-Outputs' and mimeType='application/vnd.google-apps.folder' and trashed=false"}).GetList()
            
            if len(folder_list) > 0:
                folder_id = folder_list[0]['id']
                print(f"✅ Found existing folder: {folder_list[0]['title']}")
            else:
                folder_metadata = {'title': 'AGoT-ReAct-Outputs', 'mimeType': 'application/vnd.google-apps.folder'}
                folder = drive.CreateFile(folder_metadata)
                folder.Upload()
                folder_id = folder['id']
                print(f"✅ Created new folder: AGoT-ReAct-Outputs")
            
            DRIVE_OUTPUT_DIR = folder_id  # Store folder ID for PyDrive
            DRIVE_CONNECTED = True
            print(f"✅ Google Drive connected (PyDrive2)! Folder ID: {DRIVE_OUTPUT_DIR}")
        else:
            print("⚠️ Could not authenticate with Google Drive")
            print("   Will save outputs locally only")
    
    except ImportError:
        print("⚠️ PyDrive2 not installed. Installing...")
        os.system('pip install -q pydrive2')
        print("   Please re-run this cell after installation")
    except Exception as pydrive_err:
        print(f"⚠️ PyDrive2 setup failed: {str(pydrive_err)[:100]}")
        print("   Will save outputs locally only")

def save_to_drive(filename: str, content, is_json: bool = True, append_mode: bool = False):
    """Save content to Google Drive. Falls back to local if Drive unavailable."""
    if not DRIVE_CONNECTED:
        return False
    
    try:
        # For Colab (path-based)
        if isinstance(DRIVE_OUTPUT_DIR, Path):
            drive_path = DRIVE_OUTPUT_DIR / filename
            
            if is_json:
                if append_mode and drive_path.exists():
                    with open(drive_path, 'a', encoding='utf-8') as f:
                        if isinstance(content, str):
                            f.write(content + '\n')
                        else:
                            f.write(json.dumps(content, ensure_ascii=False) + '\n')
                else:
                    with open(drive_path, 'w', encoding='utf-8') as f:
                        if isinstance(content, str):
                            f.write(content)
                        else:
                            f.write(json.dumps(content, ensure_ascii=False, indent=2))
            else:
                mode = 'a' if append_mode else 'w'
                with open(drive_path, mode, encoding='utf-8') as f:
                    f.write(content if isinstance(content, str) else str(content))
        
        # For Kaggle with PyDrive (folder ID-based)
        else:
            from pydrive2.drive import GoogleDrive
            from pydrive2.auth import GoogleAuth
            gauth = GoogleAuth()
            gauth.LoadCredentialsFile("mycreds.txt")
            drive = GoogleDrive(gauth)
            
            # Check if file exists in folder
            file_list = drive.ListFile({'q': f"title='{filename}' and '{DRIVE_OUTPUT_DIR}' in parents and trashed=false"}).GetList()
            
            if append_mode and len(file_list) > 0:
                # Append to existing file
                file_id = file_list[0]['id']
                file_obj = drive.CreateFile({'id': file_id})
                file_obj.GetContentFile(filename)
                with open(filename, 'a', encoding='utf-8') as f:
                    if is_json and not isinstance(content, str):
                        f.write(json.dumps(content, ensure_ascii=False) + '\n')
                    else:
                        f.write(str(content) + '\n')
                file_obj.SetContentFile(filename)
                file_obj.Upload()
            else:
                # Create or overwrite
                if is_json and not isinstance(content, str):
                    content = json.dumps(content, ensure_ascii=False, indent=2)
                
                with open(filename, 'w', encoding='utf-8') as f:
                    f.write(str(content))
                
                file_obj = drive.CreateFile({'title': filename, 'parents': [{'id': DRIVE_OUTPUT_DIR}]})
                file_obj.SetContentFile(filename)
                file_obj.Upload()
        
        return True
    except Exception as e:
        print(f"⚠️ Failed to save to Drive: {str(e)[:100]}")
        return False

def backup_outputs_to_drive():
    """Backup all local outputs to Google Drive (text files only)."""
    if not DRIVE_CONNECTED:
        print("⚠️ Google Drive not connected - skipping backup")
        return
    
    try:
        for local_file in OUTPUT_DIR.glob('*.jsonl'):  # Only backup JSONL files
            if local_file.is_file():
                with open(local_file, 'r', encoding='utf-8') as f:
                    content = f.read()
                save_to_drive(local_file.name, content, is_json=False)
                print(f"  ✓ Backed up {local_file.name}")
        print(f"✅ All outputs backed up to Drive")
    except Exception as e:
        print(f"⚠️ Backup failed: {str(e)[:100]}")

print("✓ Google Drive integration module loaded (Colab + Kaggle compatible)")

In [4]:
# Load GPQA Diamond
from datasets import load_dataset

print("Loading GPQA Diamond...")
gpqa_dataset = load_dataset("fingertap/GPQA-Diamond", split="test")
print(f"✓ Loaded {len(gpqa_dataset)} questions")
print(f"Fields: {gpqa_dataset.column_names}")
print(json.dumps({k: str(v)[:120] for k, v in gpqa_dataset[0].items()}, indent=2))

f:\Data Science\DS\7th Semester\ML\Project\AGoT-ReAct\Math Performance\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading GPQA Diamond...
✓ Loaded 198 questions
Fields: ['question', 'answer']
{
  "question": "Among the following exoplanets, which one has the highest density?\n\na) An Earth-mass and Earth-radius planet.\nb) A plane",
  "answer": "D"
}


## External Tools for ReAct

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
import time

# ============= External Tool Executor =============
# This is used by: Cell 12 (ReAct parsing) and Cell 15 (AGoT-ReAct solver)
# Make sure this cell runs before Cells 12 and 15

class ExternalToolExecutor:
    def __init__(self):
        self.search_history = []
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        self.max_retries = 2

    def search_wikipedia(self, entity: str) -> str:
        """Search Wikipedia with retry logic and robust error handling."""
        if not entity or len(entity.strip()) == 0:
            return "Invalid search term"
        
        for attempt in range(self.max_retries + 1):
            try:
                api_url = "https://en.wikipedia.org/w/api.php"
                params = {
                    'action': 'query',
                    'format': 'json',
                    'titles': entity.strip(),
                    'prop': 'extracts',
                    'explaintext': True,
                    'exintro': True,
                    'redirects': 1,
                    'timeout': 5
                }
                
                # Use session with timeout
                r = self.session.get(api_url, params=params, timeout=10)
                
                # Check HTTP status
                if r.status_code == 429:  # Rate limited
                    if attempt < self.max_retries:
                        time.sleep(2 ** attempt)
                        continue
                    return "Wikipedia rate limited - please try again"
                
                if r.status_code != 200:
                    return f"Wikipedia API error: {r.status_code}"
                
                # Check response content
                if not r.text or len(r.text.strip()) < 10:
                    return f"No results for '{entity}'"
                
                # Parse JSON safely
                try:
                    data = r.json()
                except Exception:
                    if attempt < self.max_retries:
                        time.sleep(1)
                        continue
                    return "Wikipedia response parsing failed"
                
                # Extract content
                pages = data.get('query', {}).get('pages', {})
                if not pages:
                    return f"No Wikipedia page found for '{entity}'"
                
                page_id = list(pages.keys())[0]
                page = pages[page_id]
                
                if 'missing' in page:
                    return f"'{entity}' not found on Wikipedia"
                
                extract = page.get('extract', '')
                if not extract:
                    return f"No content available for '{entity}'"
                
                # Truncate to first 200 words
                words = extract.split()
                snippet = ' '.join(words[:200])
                return snippet + ('...' if len(words) > 200 else '')
            
            except requests.exceptions.Timeout:
                if attempt < self.max_retries:
                    time.sleep(1)
                    continue
                return "Wikipedia search timed out"
            except requests.exceptions.ConnectionError:
                if attempt < self.max_retries:
                    time.sleep(1)
                    continue
                return "Wikipedia connection failed"
            except Exception as e:
                if attempt < self.max_retries:
                    time.sleep(1)
                    continue
                return f"Wikipedia search error: {str(e)[:50]}"
        
        return "Wikipedia search failed after retries"

    def search_web(self, query: str) -> str:
        """Search DuckDuckGo with retry logic and robust error handling."""
        if not query or len(query.strip()) == 0:
            return "Invalid search query"
        
        for attempt in range(self.max_retries + 1):
            try:
                url = f"https://html.duckduckgo.com/html/?q={quote_plus(query.strip())}"
                
                # Use session with timeout
                r = self.session.get(url, timeout=10)
                
                if r.status_code == 429:
                    if attempt < self.max_retries:
                        time.sleep(2 ** attempt)
                        continue
                    return "Search rate limited"
                
                if r.status_code != 200:
                    return f"Search API error: {r.status_code}"
                
                if not r.text or len(r.text) < 100:
                    return "Empty search response"
                
                # Parse HTML
                try:
                    soup = BeautifulSoup(r.text, features="html.parser")
                except Exception:
                    if attempt < self.max_retries:
                        time.sleep(1)
                        continue
                    return "HTML parsing failed"
                
                # Extract snippets
                snippets = []
                for item in soup.find_all("div", {"class": "result"})[:5]:
                    sn = item.find("a", {"class": "result__snippet"})
                    if sn:
                        text = sn.get_text().strip()
                        if text and len(text) > 10:
                            snippets.append(text)
                
                # Fallback to paragraphs
                if not snippets:
                    for p in soup.find_all("p", limit=3):
                        text = p.get_text().strip()
                        if len(text) > 20:
                            snippets.append(text)
                
                if not snippets:
                    return f"No web results for '{query}'"
                
                # Combine and truncate
                combined = " ".join(snippets)
                words = combined.split()
                result = ' '.join(words[:150])
                return result + ('...' if len(words) > 150 else '')
            
            except requests.exceptions.Timeout:
                if attempt < self.max_retries:
                    time.sleep(1)
                    continue
                return "Web search timed out"
            except requests.exceptions.ConnectionError:
                if attempt < self.max_retries:
                    time.sleep(1)
                    continue
                return "Web search connection failed"
            except Exception as e:
                if attempt < self.max_retries:
                    time.sleep(1)
                    continue
                return f"Web search error: {str(e)[:50]}"
        
        return "Web search failed after retries"

    def lookup_in_text(self, keyword: str, context: str) -> str:
        """Search for keyword in context text."""
        if not keyword or len(keyword.strip()) == 0:
            return "Invalid search keyword"
        
        if not context or len(context) == 0:
            return "No context available"
        
        try:
            # Split into sentences and search
            sentences = context.replace('\n', ' ').split('.')
            matches = [
                s.strip() for s in sentences 
                if keyword.lower() in s.lower() and len(s.strip()) > 5
            ]
            
            if not matches:
                return f"'{keyword}' not found in context"
            
            # Combine matches
            joined = '. '.join(matches[:3]) + '.'
            words = joined.split()
            return ' '.join(words[:150]) + ('...' if len(words) > 150 else '')
        
        except Exception as e:
            return f"Text lookup failed: {str(e)[:50]}"

external_tools = ExternalToolExecutor()
print("✓ External tools ready (Wikipedia + Web search + Text lookup)")
print("  Features: Retry logic | Timeout handling | Session management")

✓ External tools ready (Wikipedia + Web search + lookup)


## Simplified AGoT Reasoning Engine

**Architecture**: Node-based graph with Thought → Action → Observation loop

**Key Components**:
1. **AGOTNode**: state, confidence, status, action_result, domain
2. **AGOTGraph**: nodes dict, root, deduplication, path extraction
3. **Domain Analysis**: Auto-detect problem type for better reasoning
4. **Confidence Scoring**: Track reasoning quality, prune dead ends

In [ ]:
import uuid
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

# ========================================
# Simplified AGoT Node Structure
# ========================================

@dataclass
class AGOTNode:
    """Simplified AGoT node with confidence scoring."""
    id: str
    state: str  # Current reasoning state/thought
    assumptions: List[str] = field(default_factory=list)
    confidence: float = 1.0
    status: str = "open"  # "open", "dead_end", "completed"
    parents: List[str] = field(default_factory=list)
    children: List[str] = field(default_factory=list)
    depth: int = 0
    action_result: Optional[str] = None
    domain: str = ""  # Physics, Chemistry, Biology, Math

@dataclass
class AGOTGraph:
    """Graph structure for AGoT reasoning."""
    nodes: Dict[str, AGOTNode] = field(default_factory=dict)
    root_id: Optional[str] = None
    
    def add_node(self, node: AGOTNode, parent_id: Optional[str] = None) -> bool:
        """Add node to graph, merge if duplicate exists."""
        # Check for duplicate state
        duplicate = self.find_duplicate(node.state)
        if duplicate:
            # Merge: add parent link to existing node
            if parent_id and parent_id not in duplicate.parents:
                duplicate.parents.append(parent_id)
                if parent_id in self.nodes:
                    self.nodes[parent_id].children.append(duplicate.id)
            return False  # Didn't add new node
        else:
            # Add new node
            if parent_id:
                node.parents.append(parent_id)
                if parent_id in self.nodes:
                    self.nodes[parent_id].children.append(node.id)
                    node.depth = self.nodes[parent_id].depth + 1
            self.nodes[node.id] = node
            if self.root_id is None:
                self.root_id = node.id
            return True  # Added new node
    
    def find_duplicate(self, state: str) -> Optional[AGOTNode]:
        """Find node with identical state."""
        for node in self.nodes.values():
            if node.state.strip().lower() == state.strip().lower():
                return node
        return None
    
    def get_expandable_nodes(self) -> List[AGOTNode]:
        """Get all nodes with status='open'."""
        return [n for n in self.nodes.values() if n.status == "open"]
    
    def get_top_k_nodes(self, k: int = 5) -> List[AGOTNode]:
        """Get top-k nodes by confidence."""
        return sorted(self.nodes.values(), key=lambda n: n.confidence, reverse=True)[:k]
    
    def extract_solution_path(self, goal_node_id: str) -> List[AGOTNode]:
        """Extract path from root to goal node."""
        path = []
        current_id = goal_node_id
        while current_id:
            if current_id in self.nodes:
                node = self.nodes[current_id]
                path.insert(0, node)
                current_id = node.parents[0] if node.parents else None
            else:
                break
        return path
    
    def summary(self, n_nodes: int = 10) -> str:
        """Get summary of top nodes."""
        lines = []
        for node in self.get_top_k_nodes(n_nodes):
            lines.append(f"[D{node.depth}] {node.state[:80]} (conf={node.confidence:.2f}, status={node.status})")
        return "\n".join(lines)

# ========================================
# LLM Generate Function (Qwen2-7B)
# ========================================

async def llm_generate(prompt: str, temperature: float = 0.2, max_tokens: int = 512) -> str:
    """Generate from Qwen2 using HuggingFace transformers."""
    try:
        # Format prompt for Qwen2-Instruct
        messages = [
            {"role": "system", "content": "You are a helpful AI assistant skilled in PhD-level STEM reasoning."},
            {"role": "user", "content": prompt}
        ]
        
        # Use chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Tokenize and move tensors onto the model's primary device to avoid CPU/CUDA mismatches
        primary_device = next(model.parameters()).device
        inputs = tokenizer([text], return_tensors="pt", truncation=True, max_length=2048).to(primary_device)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=temperature > 0,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Decode
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        return response.strip()
        
    except Exception as e:
        print(f"⚠️ LLM error: {e}")
        return ""

# ========================================
# Domain & Problem Type Analysis
# ========================================

async def analyze_domain_and_type(question: str) -> Dict[str, str]:
    """Analyze the domain and problem type for better reasoning."""
    prompt = f"""Analyze this PhD-level question and identify:
1. Domain: Physics, Chemistry, Biology, or Math
2. Problem Type: (e.g., derivation, calculation, mechanism, pathway, proof, optimization)
3. Key Concepts: List 2-3 main concepts

Question: {question[:500]}

Format your response as:
Domain: [domain]
Type: [type]
Concepts: [concept1, concept2, concept3]"""
    
    response = await llm_generate(prompt, temperature=0.1, max_tokens=150)
    
    # Parse response
    domain = "General"
    prob_type = "Analysis"
    concepts = []
    
    for line in response.split('\n'):
        if line.startswith("Domain:"):
            domain = line.split(":", 1)[1].strip()
        elif line.startswith("Type:"):
            prob_type = line.split(":", 1)[1].strip()
        elif line.startswith("Concepts:"):
            concepts_str = line.split(":", 1)[1].strip()
            concepts = [c.strip() for c in concepts_str.split(',')]
    
    return {
        "domain": domain,
        "problem_type": prob_type,
        "key_concepts": concepts
    }

print("✓ Simplified AGoT node structure ready (with domain analysis)")

✓ AGoT graph structures & agent functions ready


In [ ]:

# ========================================
# AGoT Engine - Simplified ReAct-Style Flow with Verification
# ========================================

class AGoTEngine:
    """Simplified AGoT engine with step verification and self-consistency voting."""
    
    def __init__(self, max_depth: int = 3, top_k: int = 5, num_attempts: int = 3):
        self.max_depth = max_depth
        self.top_k = top_k
        self.num_attempts = num_attempts  # For self-consistency voting
        self.metrics = {
            "nodes_created": 0,
            "nodes_evaluated": 0,
            "nodes_pruned": 0,
            "actions_performed": 0,
            "verifications_performed": 0,
            "verifications_failed": 0
        }
    
    async def verify_action_result(self, thought: str, action_result: str, domain: str) -> Tuple[bool, str]:
        """Verify if action result is logically sound and consistent."""
        self.metrics["verifications_performed"] += 1
        
        prompt = f"""Verify if this reasoning step and its result are logically sound.

Domain: {domain}
Reasoning Step: {thought}
Result/Observation: {action_result}

Check for:
1. Logical consistency
2. Correct application of principles/formulas
3. No computational errors
4. Result makes sense in context

Respond with:
Valid: YES or NO
Reason: [brief explanation if NO]

Format:
Valid: [YES/NO]
Reason: [explanation]"""
        
        response = await llm_generate(prompt, temperature=0.1, max_tokens=200)
        
        # Parse validation
        is_valid = True
        reason = "Passed verification"
        
        for line in response.split('\n'):
            if line.startswith("Valid:"):
                valid_str = line.split(":", 1)[1].strip().upper()
                is_valid = "YES" in valid_str or "TRUE" in valid_str
            elif line.startswith("Reason:"):
                reason = line.split(":", 1)[1].strip()
        
        if not is_valid:
            self.metrics["verifications_failed"] += 1
        
        return is_valid, reason
    
    async def generate_candidate_thoughts(self, node: AGOTNode, context: str, domain_info: Dict) -> List[str]:
        """Generate candidate reasoning steps based on domain and parent context."""
        domain = domain_info.get("domain", "General")
        prob_type = domain_info.get("problem_type", "Analysis")
        
        # Build parent context from action results
        parent_context = ""
        if node.action_result:
            parent_context = f"\nPrevious observation: {node.action_result[:200]}"
        
        prompt = f"""Given the current reasoning state, generate 3-5 next logical reasoning steps.

Domain: {domain}
Problem Type: {prob_type}
Current State: {node.state}{parent_context}

Question Context: {context[:300]}

Generate specific, actionable reasoning steps (one per line):
- For Math: lemma applications, equation transforms, proof steps
- For Physics: formula applications, law applications, boundary conditions
- For Chemistry: reaction steps, intermediate predictions, mechanism steps
- For Biology: pathway interactions, regulatory steps, molecular interactions

Return ONLY the reasoning steps, one per line, no numbering:"""
        
        response = await llm_generate(prompt, temperature=0.5, max_tokens=300)
        
        # Parse candidates
        candidates = []
        for line in response.split('\n'):
            line = line.strip()
            # Remove numbering
            line = re.sub(r'^\d+[\.\)]\s*', '', line)
            line = re.sub(r'^[\-\•]\s*', '', line)
            if line and len(line) > 10:
                candidates.append(line)
        
        return candidates[:5]  # Max 5 candidates
    
    async def perform_action(self, thought: str, domain: str, context: str) -> Tuple[str, bool]:
        """Perform domain-specific action and return observation."""
        self.metrics["actions_performed"] += 1
        
        # Domain-specific action prompt
        if domain.lower() in ["math", "mathematics"]:
            action_type = "compute, derive, or verify"
        elif domain.lower() == "physics":
            action_type = "solve equation, apply law, or simulate"
        elif domain.lower() == "chemistry":
            action_type = "predict reaction, verify mechanism, or calculate"
        elif domain.lower() == "biology":
            action_type = "verify pathway, check regulation, or analyze interaction"
        else:
            action_type = "analyze or verify"
        
        prompt = f"""Execute this reasoning step and provide the result.

Action: {action_type}
Reasoning Step: {thought}
Context: {context[:200]}

Provide:
1. Result/Observation (brief, factual)
2. Success: YES or NO (whether this step leads to progress)

Format:
Result: [your observation]
Success: [YES/NO]"""
        
        response = await llm_generate(prompt, temperature=0.3, max_tokens=250)
        
        # Parse result and success
        result = response
        success = True  # Default
        
        for line in response.split('\n'):
            if line.startswith("Result:"):
                result = line.split(":", 1)[1].strip()
            elif line.startswith("Success:"):
                success_str = line.split(":", 1)[1].strip().upper()
                success = "YES" in success_str or "TRUE" in success_str
        
        return result, success
    
    def update_confidence(self, node: AGOTNode, success: bool, verification_passed: bool = True):
        """Update node confidence based on action success and verification."""
        if success and verification_passed:
            node.confidence *= 1.1
            node.confidence = min(node.confidence, 2.0)  # Cap at 2.0
        elif not verification_passed:
            # Failed verification is worse than just unsuccessful action
            node.confidence *= 0.3
            if node.confidence < 0.15:
                node.status = "dead_end"
        else:
            node.confidence *= 0.5
            if node.confidence < 0.2:
                node.status = "dead_end"
    
    def prune_low_confidence_nodes(self, graph: AGOTGraph):
        """Prune nodes with low confidence, keeping top-k."""
        all_nodes = list(graph.nodes.values())
        open_nodes = [n for n in all_nodes if n.status == "open"]
        
        if len(open_nodes) <= self.top_k:
            return  # No need to prune
        
        # Sort by confidence
        open_nodes.sort(key=lambda n: n.confidence, reverse=True)
        
        # Mark low-confidence nodes as dead_end
        for node in open_nodes[self.top_k:]:
            node.status = "dead_end"
            self.metrics["nodes_pruned"] += 1
    
    def check_goal(self, node: AGOTNode, question: str) -> bool:
        """Check if node represents a valid solution."""
        # Simple heuristic: node has high confidence and contains answer indicator
        if node.confidence < 0.7:
            return False
        
        state_lower = node.state.lower()
        result_lower = (node.action_result or "").lower()
        
        # Check for answer indicators
        answer_indicators = [
            "answer is", "solution is", "result is",
            "option a", "option b", "option c", "option d",
            "therefore a", "therefore b", "therefore c", "therefore d"
        ]
        
        for indicator in answer_indicators:
            if indicator in state_lower or indicator in result_lower:
                return True
        
        return False
    
    async def run_single_attempt(self, question: str, domain_info: Dict, attempt_num: int = 1) -> Tuple[str, AGOTGraph, Dict]:
        """Execute single AGoT reasoning attempt."""
        # Initialize graph with root node
        root = AGOTNode(
            id=str(uuid.uuid4()),
            state=f"Problem: {question[:200]}",
            domain=domain_info["domain"],
            depth=0
        )
        graph = AGOTGraph()
        graph.add_node(root)
        attempt_metrics = {"nodes_created": 1, "nodes_evaluated": 0, "actions_performed": 0}
        
        # Main AGoT loop
        for depth in range(self.max_depth):
            # Select expandable nodes
            expandable = graph.get_expandable_nodes()
            if not expandable:
                break
            
            # Limit to top-k nodes to prevent explosion
            expandable = sorted(expandable, key=lambda n: n.confidence, reverse=True)[:self.top_k]
            
            # For each expandable node
            for node in expandable:
                # Generate candidate thoughts
                candidates = await self.generate_candidate_thoughts(node, question, domain_info)
                
                # Create child nodes
                for candidate in candidates:
                    new_node = AGOTNode(
                        id=str(uuid.uuid4()),
                        state=candidate,
                        domain=domain_info["domain"],
                        assumptions=node.assumptions.copy()
                    )
                    
                    # Add to graph (will merge if duplicate)
                    added = graph.add_node(new_node, node.id)
                    if added:
                        attempt_metrics["nodes_created"] += 1
            
            # Perform actions and verify for new nodes
            new_nodes = [n for n in graph.nodes.values() if n.status == "open" and n.action_result is None]
            for node in new_nodes:
                if node.id == root.id:
                    continue  # Skip root
                
                # Perform action
                result, success = await self.perform_action(
                    node.state,
                    domain_info["domain"],
                    question
                )
                
                node.action_result = result
                attempt_metrics["nodes_evaluated"] += 1
                attempt_metrics["actions_performed"] += 1
                
                # Verify action result
                verification_passed = True
                if success:  # Only verify if action claims success
                    verification_passed, verify_reason = await self.verify_action_result(
                        node.state,
                        result,
                        domain_info["domain"]
                    )
                    if not verification_passed:
                        node.action_result += f" [VERIFICATION FAILED: {verify_reason}]"
                
                # Update confidence
                self.update_confidence(node, success, verification_passed)
            
            # Prune low-confidence nodes
            self.prune_low_confidence_nodes(graph)
            
            # Check if any node reached goal
            goal_nodes = [n for n in graph.nodes.values() if self.check_goal(n, question)]
            if goal_nodes:
                # Sort by confidence and mark best as completed
                goal_nodes.sort(key=lambda n: n.confidence, reverse=True)
                goal_nodes[0].status = "completed"
                break
        
        # Extract solution
        completed_nodes = [n for n in graph.nodes.values() if n.status == "completed"]
        if completed_nodes:
            best_node = max(completed_nodes, key=lambda n: n.confidence)
            solution_path = graph.extract_solution_path(best_node.id)
            final_answer = await self.synthesize_solution(solution_path, question)
        else:
            # Fallback: use highest confidence node
            top_nodes = graph.get_top_k_nodes(1)
            if top_nodes:
                solution_path = graph.extract_solution_path(top_nodes[0].id)
                final_answer = await self.synthesize_solution(solution_path, question)
            else:
                final_answer = "Unable to determine answer"
        
        return final_answer, graph, attempt_metrics
    
    async def run(self, question: str) -> Tuple[str, AGOTGraph, Dict]:
        """Execute AGoT reasoning with self-consistency voting."""
        # Step 1: Analyze domain and problem type
        domain_info = await analyze_domain_and_type(question)
        
        # Step 2: Run multiple independent attempts
        attempts = []
        all_graphs = []
        
        print(f"  Running {self.num_attempts} independent reasoning attempts...")
        for i in range(self.num_attempts):
            try:
                final_answer, graph, attempt_metrics = await self.run_single_attempt(
                    question, domain_info, attempt_num=i+1
                )
                attempts.append({
                    "answer": final_answer,
                    "graph": graph,
                    "metrics": attempt_metrics
                })
                all_graphs.append(graph)
                
                # Update global metrics
                for key in attempt_metrics:
                    if key in self.metrics:
                        self.metrics[key] += attempt_metrics[key]
                
            except Exception as e:
                print(f"  ⚠️ Attempt {i+1} failed: {str(e)[:60]}")
                continue
        
        # Step 3: Vote on final answer (self-consistency)
        if not attempts:
            return "Unable to determine answer", AGOTGraph(), self.metrics
        
        # Extract all answers and vote
        answers = [extract_choice_letter(a["answer"], "?") for a in attempts]
        answer_counts = {}
        for ans in answers:
            answer_counts[ans] = answer_counts.get(ans, 0) + 1
        
        # Get majority vote
        voted_answer = max(answer_counts.items(), key=lambda x: x[1])[0]
        vote_confidence = answer_counts[voted_answer] / len(answers)
        
        # Find best graph that produced the voted answer
        best_graph = None
        best_confidence = 0
        for attempt in attempts:
            attempt_answer = extract_choice_letter(attempt["answer"], "?")
            if attempt_answer == voted_answer:
                # Get top node confidence from this graph
                top_nodes = attempt["graph"].get_top_k_nodes(1)
                if top_nodes and top_nodes[0].confidence > best_confidence:
                    best_confidence = top_nodes[0].confidence
                    best_graph = attempt["graph"]
        
        # Use best graph or first graph as fallback
        final_graph = best_graph if best_graph else all_graphs[0]
        
        # Create final answer with voting info
        final_answer = f"The answer is {voted_answer} (voted {answer_counts[voted_answer]}/{len(answers)} attempts, confidence={vote_confidence:.1%})"
        
        self.metrics["voting_confidence"] = vote_confidence
        self.metrics["vote_distribution"] = answer_counts
        
        return final_answer, final_graph, self.metrics
    
    async def synthesize_solution(self, path: List[AGOTNode], question: str) -> str:
        """Synthesize final solution from reasoning path."""
        path_summary = "\n".join([
            f"Step {i+1}: {node.state}\n  → {node.action_result or 'N/A'}"
            for i, node in enumerate(path) if node.action_result
        ])
        
        prompt = f"""Based on this reasoning path, provide the final answer.

Question: {question}

Reasoning Path:
{path_summary[:800]}

IMPORTANT: End your response with exactly one of these:
- "The answer is A"
- "The answer is B"  
- "The answer is C"
- "The answer is D"

Provide brief justification then state the answer:"""
        
        response = await llm_generate(prompt, temperature=0.2, max_tokens=300)
        return response.strip()

print("✓ AGoT engine ready with step verification + self-consistency voting")
print("  Features: Verify each action | Vote across 3 independent attempts")

✓ AGoT engine ready (layer-based with graph evaluation & pruning)


## ReAct Verification

In [ ]:
# ============= ReAct Action Parsing Helpers =============
# DEPENDENCY: Cell 8 (ExternalToolExecutor) must run before this

def parse_action(text: str) -> tuple:
    """Parse action from text: search[...], lookup[...], calculate[...], finish[A/B/C/D]"""
    patterns = [
        (r"search\[(.+?)\]", "search"),
        (r"lookup\[(.+?)\]", "lookup"),
        (r"calculate\[(.+?)\]", "calculate"),
        (r"finish\[([A-D])\]", "finish"),
    ]
    t = text.lower()
    for pattern, action_type in patterns:
        m = re.search(pattern, t, re.IGNORECASE | re.DOTALL)
        if m:
            return action_type, m.group(1).strip()
    return None, None


def normalize_choice_letter(text: str, fallback: str = "?") -> str:
    """Extract a top-level A/B/C/D letter from free text; fallback if none."""
    if not text:
        return fallback
    patterns = [
        r"answer\s+is\s+([A-D])",
        r"option\s+([A-D])",
        r"([A-D])\)",
        r"\b([A-D])\b",
    ]
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            letter = m.group(1).upper()
            if letter in ["A", "B", "C", "D"]:
                return letter
    return fallback


def extract_choice_letter(text: str, fallback: str = "?") -> str:
    """Robustly extract A/B/C/D from messy LLM output with priority patterns."""
    if not text:
        return fallback
    
    text_upper = text.upper()
    
    # Priority-ordered patterns for extracting choice
    patterns = [
        r"ANSWER\s+IS\s+([A-D])",
        r"ANSWER\s*[:=]\s*([A-D])",
        r"FINAL\s+ANSWER\s*[:=]?\s*([A-D])",
        r"CHOICE\s+([A-D])",
        r"OPTION\s+([A-D])",
        r"THEREFORE\s+([A-D])",
        r"([A-D])\s*[\.\):\-]",
        r"\b([A-D])\b(?=[\s\.\,\!\?]|$)",
    ]
    
    for pat in patterns:
        m = re.search(pat, text_upper)
        if m:
            letter = m.group(1).upper()
            if letter in ["A", "B", "C", "D"]:
                return letter
    
    # Last resort: find any standalone letter A-D
    last_match = re.findall(r"[A-D]", text_upper)
    if last_match:
        for letter in reversed(last_match):
            if letter in ["A", "B", "C", "D"]:
                return letter
    
    return fallback

print("✓ ReAct helpers ready (parse_action, normalize_choice_letter, extract_choice_letter)")

✓ ReAct verification ready


## AGoT + ReAct Solver

In [ ]:
import asyncio

# Helper: robustly extract A/B/C/D from text
def extract_choice_letter(text: str, fallback: str = "?") -> str:
    if not text:
        return fallback
    
    text_upper = text.upper()
    
    # Priority-ordered patterns for extracting choice
    patterns = [
        r"ANSWER\s+IS\s+([A-D])",
        r"ANSWER\s*[:=]\s*([A-D])",
        r"FINAL\s+ANSWER\s*[:=]?\s*([A-D])",
        r"CHOICE\s+([A-D])",
        r"OPTION\s+([A-D])",
        r"THEREFORE\s+([A-D])",
        r"([A-D])\s*[\.\):\-]",
        r"\b([A-D])\b(?=[\s\.\,\!\?]|$)",
    ]
    
    for pat in patterns:
        m = re.search(pat, text_upper)
        if m:
            letter = m.group(1).upper()
            if letter in ["A", "B", "C", "D"]:
                return letter
    
    # Last resort: find any standalone letter A-D
    last_match = re.findall(r"[A-D]", text_upper)
    if last_match:
        for letter in reversed(last_match):
            if letter in ["A", "B", "C", "D"]:
                return letter
    
    return fallback


async def agot_react_solve_question(example: dict, agot_engine: AGoTEngine) -> dict:
    """Solve question using INTERLEAVED AGoT(Thinking)->Action->Observation loop."""
    question = example.get('question', '')
    correct_answer = example.get('correct_answer', '')
    index = example.get('index', -1)

    try:
        print(f"\n[Q{index}] Starting AGoT-ReAct INTERLEAVED LOOP...")
        
        # Step 1: Run AGoT (generates reasoning graph)
        print(f"[Q{index}] Step 1: AGoT REASONING (building reasoning graph)...")
        try:
            agot_final, agot_graph, agot_metrics = await agot_engine.run(question)
            agot_answer = extract_choice_letter(agot_final, "?")
            print(f"[Q{index}]   → AGoT hypothesis: {agot_answer} (confidence: {agot_metrics.get('voting_confidence', 0):.1%})")
        except Exception as e:
            print(f"⚠️ AGoT failed on Q{index}: {str(e)[:80]}")
            agot_final = f"AGoT failed: {str(e)[:100]}"
            agot_graph = AGOTGraph()
            agot_metrics = {'nodes_created': 0, 'nodes_evaluated': 0}
            agot_answer = "?"

        # Step 2: INTERLEAVED LOOP: Extract AGoT Thinking -> ReAct Action -> Observation
        combined_trace = []
        current_answer = agot_answer
        
        print(f"[Q{index}] Entering interleaved loop ({REACT_MAX_STEPS} iterations)...")
        
        for step_num in range(1, REACT_MAX_STEPS + 1):
            # Step A: EXTRACT AGoT THINKING (from graph reasoning paths)
            print(f"[Q{index}] Step {step_num+1}a: AGoT THINKING (from reasoning graph)...")
            
            # Get top-k reasoning nodes from AGoT graph
            top_nodes = agot_graph.get_top_k_nodes(k=3)
            agot_thinking = ""
            
            if top_nodes:
                # Extract reasoning paths from graph
                for i, node in enumerate(top_nodes, 1):
                    reasoning_step = f"{i}. {node.state}"
                    if node.action_result:
                        reasoning_step += f" → {node.action_result[:100]}"
                    agot_thinking += reasoning_step + "\n"
            else:
                agot_thinking = "No reasoning paths available from AGoT"
            
            combined_trace.append({
                "step": step_num,
                "phase": "AGoT_Thinking",
                "content": agot_thinking[:200],
                "iteration": step_num
            })
            print(f"[Q{index}]   → Thinking: {agot_thinking[:80]}...")
            
            # Step B: ReAct ACTION (based on AGoT thinking)
            print(f"[Q{index}] Step {step_num+1}b: ReAct ACTION (guided by AGoT thinking + accumulated evidence)...")
            
            # BUILD ACCUMULATED EVIDENCE from previous iterations
            accumulated_evidence = ""
            prior_observations = [t for t in combined_trace if t.get('phase') == 'ReAct_Observation']
            if prior_observations:
                accumulated_evidence += "EVIDENCE GATHERED SO FAR:\n"
                for i, obs in enumerate(prior_observations, 1):
                    evidence_snippet = obs.get('content', '')[:120]
                    accumulated_evidence += f"  {i}. {evidence_snippet}\n"
            
            # ACTION PROMPT with accumulated evidence
            action_prompt = (
                f"Based on AGoT reasoning and evidence gathered so far, execute ONE verification action:\n"
                f"Question: {question}\n"
                f"Current hypothesis: {current_answer}\n"
                f"AGoT reasoning paths:\n{agot_thinking[:300]}\n"
                f"{accumulated_evidence}\n"
                f"Action: <search[term] | lookup[keyword] | calculate[expression] | finish[A/B/C/D]>"
            )
            
            action_text = await llm_generate(action_prompt, temperature=0.2, max_tokens=80)
            action_type, parameter = parse_action(action_text)
            
            # Step C: ReAct OBSERVATION (result of action)
            print(f"[Q{index}] Step {step_num+1}c: OBSERVATION...")
            if action_type == "finish":
                final = normalize_choice_letter(parameter, current_answer)
                observation = f"Finish with answer {final}."
                current_answer = final
            elif action_type == "search":
                observation = external_tools.search_wikipedia(parameter)
            elif action_type == "lookup":
                prev_obs = [t for t in combined_trace if t.get('phase') == 'ReAct_Observation']
                observation = external_tools.lookup_in_text(parameter, prev_obs[-1].get('content', '') if prev_obs else "")
            elif action_type == "calculate":
                try:
                    result = eval(parameter, {"__builtins__": {}}, {})
                    observation = f"Calculation result: {parameter} = {result}"
                except Exception as e:
                    observation = f"Calculation error: {str(e)[:100]}"
            else:
                observation = "No valid action. Use search[], lookup[], calculate[], or finish[]."
            
            combined_trace.append({
                "step": step_num,
                "phase": "ReAct_Action",
                "action": f"{action_type}[{parameter}]" if action_type else action_text,
                "iteration": step_num
            })
            combined_trace.append({
                "step": step_num,
                "phase": "ReAct_Observation",
                "content": observation[:300],
                "iteration": step_num
            })
            
            print(f"[Q{index}]   → Action: {action_type}[{parameter if parameter else '?'}]")
            print(f"[Q{index}]   → Observation: {observation[:100]}...")
            
            if action_type == "finish":
                print(f"[Q{index}] ✓ Loop finished at iteration {step_num}")
                break
        
        # Build final trace
        trace_lines = [
            "=== INTERLEAVED: AGoT(Thinking) → ReAct(Action) → Observation ===",
            f"Question: {question[:100]}...",
            f"Domain: {agot_graph.nodes[agot_graph.root_id].domain if agot_graph.root_id else 'Unknown'}",
            "",
            "INITIAL AGoT REASONING GRAPH:",
            f"  Voting: {agot_metrics.get('vote_distribution', {})} (confidence: {agot_metrics.get('voting_confidence', 0):.1%})",
            f"  Nodes created: {agot_metrics.get('nodes_created', 0)} | Evaluated: {agot_metrics.get('nodes_evaluated', 0)}",
            f"  Initial answer: {agot_answer}",
            ""
        ]
        
        for t in combined_trace:
            if t['phase'] == 'AGoT_Thinking':
                trace_lines.append(f"Iteration {t['step']}a) AGoT THINKING (from reasoning graph):")
                trace_lines.append(f"  {t['content'][:150]}")
            elif t['phase'] == 'ReAct_Action':
                trace_lines.append(f"Iteration {t['step']}b) ReAct ACTION:")
                trace_lines.append(f"  {t['action']}")
            elif t['phase'] == 'ReAct_Observation':
                trace_lines.append(f"Iteration {t['step']}c) OBSERVATION:")
                trace_lines.append(f"  {t['content'][:150]}")
        
        trace_lines.append(f"\n=== FINAL ANSWER: {current_answer} ===")
        trace_lines.append(f"Correct Answer: {correct_answer}")
        trace_lines.append(f"Result: {'✓ CORRECT' if current_answer == correct_answer else '✗ INCORRECT'}")
        
        trace = "\n".join(trace_lines)

        return {
            "index": index,
            "question": question,
            "correct_answer": correct_answer,
            "react_answer": current_answer if current_answer != "?" else agot_answer,
            "is_correct": (current_answer if current_answer != "?" else agot_answer) == correct_answer,
            "react_trace": trace,
            "steps": {
                "combined_trace": combined_trace,  # Interleaved trace
                "agot_metrics": agot_metrics
            },
            "agot_answer": agot_answer,
            "voting_info": {
                "distribution": agot_metrics.get('vote_distribution', {}),
                "confidence": agot_metrics.get('voting_confidence', 0)
            }
        }

    except Exception as e:
        print(f"⚠️ Critical error solving Q{index}: {str(e)[:100]}")
        return {
            "index": index,
            "question": question,
            "correct_answer": correct_answer,
            "react_answer": "?",
            "is_correct": False,
            "react_trace": f"CRITICAL ERROR: {str(e)[:300]}",
            "steps": {"combined_trace": [], "agot_metrics": {}},
            "agot_answer": "?",
        }

# Initialize AGoT engine with self-consistency voting
agot_engine = AGoTEngine(max_depth=AGOT_MAX_DEPTH, top_k=AGOT_TOP_K, num_attempts=AGOT_NUM_ATTEMPTS)
print(f"✓ AGoT-ReAct solver ready (INTERLEAVED structure)")
print(f"✓ Flow: AGoT(builds reasoning graph) → Loop(Extract thinking → Action → Observation)")
print(f"✓ Using Qwen2-7B-Instruct with:")
print(f"  - AGoT reasoning graph extraction for thinking")
print(f"  - ReAct action selection & external verification")
print(f"  - Self-consistency voting ({AGOT_NUM_ATTEMPTS} attempts)")

✓ AGoT+ReAct solver ready (lmax=2, nmax=3)


## Batch Evaluation with Checkpoints

In [ ]:
# ============= CONFIGURATION: Set row range to run =============
# Set these to run a specific range of questions (useful for parallel sessions)
# Examples:
#   ROWS_START = 0, ROWS_END = 50      # Run questions 0-49 in session 1
#   ROWS_START = 50, ROWS_END = 100    # Run questions 50-99 in session 2
#   ROWS_START = None, ROWS_END = None # Run ALL questions (default)

ROWS_START = None  # Start index (inclusive) - set to 0 to start from beginning
ROWS_END = None    # End index (exclusive) - set to 50 to stop at question 49

# Auto-detect if running all or range
if ROWS_START is None or ROWS_END is None:
    print("🔄 MODE: Running ALL questions")
    ROWS_START = 0
    ROWS_END = None  # Will be set to len(formatted_data) below
else:
    print(f"🎯 MODE: Running questions {ROWS_START}-{ROWS_END-1} (parallel segment)")

# ============= Prepare data with validation =============
formatted_data = []
for idx, ex in enumerate(gpqa_dataset):
    q = ex.get('question', '')
    ans = (ex.get('answer','') or '').strip().upper()
    if len(ans) > 1:
        m = re.search(r'([A-D])', ans)
        if m:
            ans = m.group(1)
    # Validate fields exist
    if not q or not ans:
        print(f"⚠️ Skipping invalid example at index {idx}")
        continue
    formatted_data.append({
        'index': idx,
        'question': q,
        'correct_answer': ans
    })
print(f"Prepared {len(formatted_data)} examples")

# Load checkpoint
checkpoint_data = {"evaluated_indices": set(), "accumulated_results": []}
if GPQA_CHECKPOINT_PATH.exists():
    try:
        with open(GPQA_CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
            saved = json.load(f)
            checkpoint_data['evaluated_indices'] = set(saved.get('evaluated_indices', []))
            checkpoint_data['accumulated_results'] = saved.get('accumulated_results', [])
        print(f"✓ Checkpoint: {len(checkpoint_data['evaluated_indices'])} already evaluated")
    except Exception:
        print("⚠️ Checkpoint corrupted, starting fresh")

# Verify AGoT engine exists (dependency check)
try:
    assert agot_engine is not None
    print(f"✓ AGoT engine verified: {AGOT_NUM_ATTEMPTS} attempts, max_depth={AGOT_MAX_DEPTH}")
except NameError:
    print("❌ ERROR: AGoT engine not initialized. Make sure Cell 11 ran successfully.")
    raise

# Compute remaining questions
all_indices = set(range(len(formatted_data)))
remaining = sorted(all_indices - checkpoint_data['evaluated_indices'])

# Set end index if not specified
if ROWS_END is None:
    ROWS_END = len(formatted_data)

# Validate range
if ROWS_START < 0 or ROWS_END > len(formatted_data) or ROWS_START >= ROWS_END:
    print(f"❌ ERROR: Invalid range [{ROWS_START}, {ROWS_END}). Valid range: [0, {len(formatted_data)})")
    raise ValueError(f"Invalid row range: ROWS_START={ROWS_START}, ROWS_END={ROWS_END}")

# Get batch indices for this session (excluding already evaluated)
all_batch = set(range(ROWS_START, ROWS_END))
batch_indices = sorted(all_batch - checkpoint_data['evaluated_indices'])

# Summary
print(f"\n{'='*70}")
print(f"📊 SESSION CONFIGURATION")
print(f"{'='*70}")
print(f"Running range: questions {ROWS_START}-{ROWS_END-1} ({ROWS_END-ROWS_START} total)")
print(f"Already evaluated in this range: {len(all_batch & checkpoint_data['evaluated_indices'])}")
print(f"To evaluate in this session: {len(batch_indices)}")
print(f"Total progress: {len(checkpoint_data['evaluated_indices'])}/{len(formatted_data)}")
if batch_indices:
    print(f"Question indices to process: {batch_indices[0]}-{batch_indices[-1]}")
else:
    print(f"✅ All questions in this range already evaluated!")
print(f"{'='*70}\n")

# Initialize results
results = []

if not batch_indices:
    print("✓ All examples in this range already evaluated!")
    # Load results for this range if needed
    results = [r for r in checkpoint_data['accumulated_results'] if r['index'] >= ROWS_START and r['index'] < ROWS_END]
else:
    
    # Run async batch
    async def run_batch():
        batch_results = []
        for idx in tqdm(batch_indices, desc="AGoT+ReAct"):
            try:
                result = await agot_react_solve_question(formatted_data[idx], agot_engine)
                batch_results.append(result)
                checkpoint_data['evaluated_indices'].add(idx)
                checkpoint_data['accumulated_results'].append(result)
                
                # Save incrementally to local files
                with open(GPQA_OUTPUT_PATH, 'a', encoding='utf-8') as f:
                    json.dump({
                        "index": result['index'],
                        "question": result['question'],
                        "answer": result['react_answer'],
                        "correct_answer": result['correct_answer'],
                        "is_correct": result['is_correct'],
                        "react_trace": result['react_trace'],
                        "timestamp": datetime.now().isoformat()
                    }, f, ensure_ascii=False)
                    f.write("\n")
                
                with open(GPQA_TRACES_PATH, 'a', encoding='utf-8') as f:
                    json.dump(result, f, ensure_ascii=False)
                    f.write("\n")
                
                # BACKUP TO GOOGLE DRIVE
                if DRIVE_CONNECTED:
                    # Save to Drive incrementally
                    save_to_drive('gpqa_agot_react_results.jsonl', {
                        "index": result['index'],
                        "question": result['question'],
                        "answer": result['react_answer'],
                        "correct_answer": result['correct_answer'],
                        "is_correct": result['is_correct'],
                        "react_trace": result['react_trace'],
                        "timestamp": datetime.now().isoformat()
                    }, is_json=True, append_mode=True)
                    
                    # Save detailed traces to Drive
                    save_to_drive('gpqa_agot_react_detailed_traces.jsonl', result, is_json=True, append_mode=True)
                
                # Save checkpoint after EACH question
                checkpoint_obj = {
                    'evaluated_indices': sorted(list(checkpoint_data['evaluated_indices'])),
                    'accumulated_results': checkpoint_data['accumulated_results'][-50:],  # keep last 50
                    'timestamp': datetime.now().isoformat()
                }
                
                with open(GPQA_CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
                    json.dump(checkpoint_obj, f, ensure_ascii=False, indent=2)
                
                # Backup checkpoint to Drive
                if DRIVE_CONNECTED:
                    save_to_drive('gpqa_agot_checkpoint.json', checkpoint_obj, is_json=True, append_mode=False)
                
                print("Printing to keep the session active in kaggle...")
            except Exception as e:
                print(f"⚠️ Error on index {idx}: {str(e)[:100]}")
                continue
        
        return batch_results
    
    # Execute - wrap in asyncio.run() for Jupyter/Kaggle compatibility
    try:
        # Try IPython's native async support first
        results = await run_batch()
    except Exception:
        # Fallback to asyncio.run() if await doesn't work at top level
        results = asyncio.run(run_batch())
    
    if results:
        correct_count = sum(1 for r in results if r['is_correct'])
        batch_accuracy = correct_count / len(results) * 100
        print(f"\n✓ Session complete: {correct_count}/{len(results)} correct ({batch_accuracy:.1f}%)")
    else:
        print("⚠️ No results generated")

print(f"\nFinal status: {len(checkpoint_data['evaluated_indices'])}/{len(formatted_data)} total questions evaluated")

NameError: name 'GPQA_CHECKPOINT_PATH' is not defined

## Metrics & Analysis

In [ ]:
# Calculate metrics
results_df = pd.DataFrame(results)
correct_results = results_df[results_df['is_correct'] == True]
incorrect_results = results_df[results_df['is_correct'] == False]

batch_correct = len(correct_results)
total_batch = len(results_df)
batch_accuracy = batch_correct / total_batch * 100 if total_batch else 0

print("\n" + "="*60)
print("BATCH ANALYSIS")
print("="*60)
print(f"Correct: {batch_correct}/{total_batch} ({batch_accuracy:.1f}%)")

# Diagnostic: Count "?" answers
unknown_answers = results_df[results_df['react_answer'] == '?']
unknown_agot = results_df[results_df['agot_answer'] == '?']
print(f"\nDiagnostics:")
print(f"  Questions with ReAct '?' answers: {len(unknown_answers)}/{total_batch} ({len(unknown_answers)/total_batch*100:.1f}%)")
print(f"  Questions with AGoT '?' answers: {len(unknown_agot)}/{total_batch} ({len(unknown_agot)/total_batch*100:.1f}%)")

if len(unknown_answers) > 0:
    print(f"\nSample questions with '?' ReAct answers (first 3):")
    for _, row in unknown_answers.head(3).iterrows():
        print(f"  Q: {row['question'][:80]}...")
        print(f"    AGoT: {row['agot_answer']} | ReAct: {row['react_answer']} | Gold: {row['correct_answer']}")
        print(f"    Trace: {row['react_trace'][:150]}...")

if len(incorrect_results) > 0:
    print("\nSample incorrect (first 3):")
    for _, row in incorrect_results.head(3).iterrows():
        print(f"  Q: {row['question'][:90]}...")
        print(f"  Model: {row['react_answer']} | Gold: {row['correct_answer']}")

# Cumulative metrics
all_eval = len(checkpoint_data['evaluated_indices'])
cumulative_stats = {'total_all_batches': all_eval, 'correct_all_batches': 0, 'batches_completed': 0}
if GPQA_CUMULATIVE_PATH.exists():
    try:
        with open(GPQA_CUMULATIVE_PATH, 'r') as f:

In [ ]:

print("\n" + "="*60)
print("🔄 BACKUP: Syncing all outputs to Google Drive...")
print("="*60)

if DRIVE_CONNECTED:
    # Save batch metrics
    batch_metrics = {
        'batch_correct': batch_correct,
        'total_batch': total_batch,
        'batch_accuracy': batch_accuracy,
        'timestamp': datetime.now().isoformat(),
        'unknown_answers_count': len(unknown_answers),
        'unknown_agot_count': len(unknown_agot)
    }
    save_to_drive('batch_metrics.json', batch_metrics, is_json=True, append_mode=False)
    print(f"  ✓ Batch metrics saved")
    
    # Backup checkpoint
    backup_outputs_to_drive()
    
    print(f"\n✅ SUCCESS: All outputs are safely backed up to Google Drive!")
    print(f"📁 Drive location: {DRIVE_OUTPUT_DIR}")
    print(f"   Files saved:")
    print(f"   - gpqa_agot_react_results.jsonl")
    print(f"   - gpqa_agot_react_detailed_traces.jsonl")
    print(f"   - gpqa_agot_checkpoint.json")
    print(f"   - batch_metrics.json")
else:
    print("\n⚠️ Google Drive not connected.")
    print("📝 Local files saved to:")
    print(f"   {OUTPUT_DIR}")
    print("\n💡 To save to Drive later:")
    print(f"   1. Download files from {OUTPUT_DIR}")
    print(f"   2. Upload to your Google Drive manually")
    print(f"   3. Or re-run this notebook in Google Colab with Drive mounting enabled")